# Tableau → Fabric: VizQL Data Service Bridge

This notebook demonstrates **Play 1** of the Tableau + Microsoft Fabric AI Bridge:
pulling a complete, governed Tableau data source into a Fabric Lakehouse as a Delta table
using Tableau's VizQL Data Service (VDS) REST API — without migrating or replacing
anything in the Tableau environment.

---

**Architecture**
```
Tableau Cloud (governed data source)
        ↓  VizQL Data Service REST API
Fabric Notebook (Python / PySpark)
        ↓  spark.createDataFrame()
Fabric Lakehouse (Delta table)
        ↓
SQL Analytics Endpoint / Power BI / Fabric Data Agent
```

---

**Prerequisites**
- Tableau Cloud or Tableau Server 2025.1+
- Creator license on the Tableau site
- Personal Access Token (PAT) stored in Azure Key Vault
- Fabric Lakehouse attached to this notebook

---

**Cells in this notebook**
1. Configuration
2. Authenticate to Tableau
3. Discover the data source
4. Read schema metadata
5. Query the full dataset
6. Sanitize column names for Delta
7. Write to Delta table in Lakehouse

## ⚠️ Start Here — Plug In Your Variables

Before running this notebook, fill in the following values in **Cell 1 (Configuration)**:

| Variable | Where to find it |
|----------|------------------|
| `PAT_NAME` | The name you gave your PAT in Tableau Cloud account settings |
| `POD` | First part of your Tableau Cloud URL (e.g. `10ay.online.tableau.com`) |
| `SITE` | The site slug from your Tableau Cloud URL (e.g. `mysite`) |
| `DATASOURCE_SEARCH` | Partial name of your published data source |
| `TABLE_NAME` | Name for the Delta table to create in your Lakehouse |
| Key Vault URL | Your Azure Key Vault URL in the `getSecret` call |
| Secret name | The name of your PAT secret in Key Vault |

**Also make sure:**
- Your Fabric Lakehouse is attached to this notebook (Explorer pane → Add lakehouse)
- Your Fabric workspace managed identity has Key Vault Secrets User access

See `Play1_Runbook.docx` for full setup instructions.


## Cell 1 — Configuration

Set your Tableau environment details here. The PAT secret is retrieved securely from
Azure Key Vault using `notebookutils.credentials.getSecret` — no credentials are
hardcoded in this notebook.

**To adapt this notebook to a different data source**, update:
- `DATASOURCE_SEARCH` — the name (or partial name) of the published data source to find
- `TABLE_NAME` — the Delta table name to create in the Lakehouse

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────────────────────
PAT_NAME          = ""                     # Your PAT name from Tableau account settings
POD               = ""                     # Your Tableau Cloud pod (e.g. 10ay.online.tableau.com)
SITE              = ""                     # Your site contentUrl slug
DATASOURCE_SEARCH = ""                     # Partial name of your published data source
TABLE_NAME        = ""                     # Delta table name to create in your Lakehouse

# Retrieve PAT secret securely from Azure Key Vault
# Replace with your Key Vault URL and secret name
PAT_SECRET = notebookutils.credentials.getSecret(
    "https://<your-keyvault-name>.vault.azure.net/",
    "<your-secret-name>"
)

BASE = f"https://{POD}"
print("✓ Configuration loaded")
print(f"  Pod:          {POD}")
print(f"  Site:         {SITE}")
print(f"  Data source:  {DATASOURCE_SEARCH}")
print(f"  Target table: {TABLE_NAME}")
print(f"  PAT secret:   retrieved from Key Vault ✓")


## Cell 2 — Authenticate to Tableau

Tableau's REST API uses a session token for all subsequent requests. We authenticate
using a Personal Access Token (PAT), which is the recommended approach for automated
workflows — it avoids hardcoded username/password and can be revoked independently.

The signin endpoint returns:
- `token` — used as the `X-Tableau-Auth` header on every subsequent API call
- `site.id` — the internal site GUID needed to scope REST API calls to the right site

**Note:** If you get a 401 or 404 error on a later cell, re-run this cell to get a
fresh token before continuing. Tokens expire after 15 days of inactivity, and
concurrent sessions with the same PAT will invalidate each other.

In [ ]:
import requests
import json
import pandas as pd

print("Authenticating to Tableau Cloud...")

auth_response = requests.post(
    f"{BASE}/api/3.24/auth/signin",
    json={
        "credentials": {
            "personalAccessTokenName": PAT_NAME,
            "personalAccessTokenSecret": PAT_SECRET,
            "site": {"contentUrl": SITE}
        }
    },
    headers={"Content-Type": "application/json", "Accept": "application/json"}
)
auth_response.raise_for_status()

auth_data = auth_response.json()
token   = auth_data["credentials"]["token"]
site_id = auth_data["credentials"]["site"]["id"]

HEADERS = {
    "X-Tableau-Auth": token,
    "Content-Type": "application/json",
    "Accept": "application/json"
}

print(f"✓ Authenticated successfully")
print(f"  Token:    {token[:20]}...")
print(f"  Site ID:  {site_id}")

## Cell 3 — Discover the Data Source

Rather than hardcoding a data source LUID (Locally Unique Identifier), we discover it
dynamically by listing all published data sources on the site and matching by name.

This makes the notebook portable — change `DATASOURCE_SEARCH` in the config cell
and it will find the right data source automatically.

The LUID is the key used by all VDS API calls to identify which data source to query.

In [ ]:
print(f"Searching for data source matching '{DATASOURCE_SEARCH}'...")

ds_response = requests.get(
    f"{BASE}/api/3.24/sites/{site_id}/datasources",
    headers=HEADERS
)
ds_response.raise_for_status()

datasources = ds_response.json().get("datasources", {}).get("datasource", [])
if isinstance(datasources, dict):
    datasources = [datasources]

print(f"  {len(datasources)} data source(s) found on site:")
for ds in datasources:
    print(f"   - {ds['name']} (id: {ds['id']})")

target_ds = next(
    (ds for ds in datasources if DATASOURCE_SEARCH.lower() in ds["name"].lower()), None
)
if not target_ds:
    raise ValueError(f"No data source matching '{DATASOURCE_SEARCH}' found. Check the site.")

datasource_luid = target_ds["id"]
print(f"\n✓ Matched: {target_ds['name']}")
print(f"  LUID:    {datasource_luid}")
print(f"  Type:    {target_ds.get('type', 'unknown')}")
print(f"  Project: {target_ds.get('project', {}).get('name', 'unknown')}")

## Cell 4 — Read Schema Metadata

The VDS `read-metadata` endpoint returns the full field catalog for the data source —
every field available to query, its data type, and its column class
(raw column, calculated field, bin, group, etc.).

This is useful for:
- Understanding what fields exist before writing a query
- Identifying which fields are dimensions vs measures
- Discovering calculated fields that exist in the Tableau data source
- **Future Play 3**: feeding this schema into OneLake Catalog as registered metadata

The `fieldCaption` values from this output are exactly what you pass into VDS queries.

**Important:** Fields from secondary/joined tables will cause VDS to perform an inner
join — only returning rows that match in both tables. For Superstore, the `Returned`
field comes from a secondary Returns table and reduces the dataset from ~10,000 rows
to ~800. Only include fields from the primary table unless you want that join behavior.

In [ ]:
print("Reading data source schema from VDS metadata endpoint...\n")

meta_response = requests.post(
    f"{BASE}/api/v1/vizql-data-service/read-metadata",
    json={"datasource": {"datasourceLuid": datasource_luid}},
    headers=HEADERS
)
meta_response.raise_for_status()
fields = meta_response.json().get("data", [])

print(f"✓ {len(fields)} fields available:\n")
print(f"  {'Class':<14} {'Field Name':<40} {'Data Type'}")
print(f"  {'─'*14} {'─'*40} {'─'*12}")
for f in fields:
    print(f"  [{f.get('columnClass', '?'):<12}]  {f['fieldCaption']:<40}  {f['dataType']}")

field_captions = [f["fieldCaption"] for f in fields if f.get("columnClass") == "COLUMN"]
print(f"\n  Raw columns available to query: {len(field_captions)}")

## Cell 5 — Query the Full Dataset

The VDS `query-datasource` endpoint returns actual data from the published data source.
Unlike a Tableau visualization which aggregates by default, VDS lets us request raw
row-level data by omitting the `function` parameter on each field.

**Field selection note:** `Returned` is intentionally excluded from this query.
It belongs to a secondary Returns table in the Superstore data model — including it
causes VDS to perform an inner join, reducing the result set from ~10,000 rows to ~800.
All other fields are from the primary Orders table and return the full dataset.

**Note on VDS rate limits:** Each Creator license on the Tableau site adds 100 queries/hour
to the site-wide cap. For high-frequency pipelines, design queries to pull at the
right grain rather than making many small calls.

In [ ]:
print("Querying full dataset from Tableau VDS...\n")

# All fields from the primary Orders table
# Note: 'Returned' is intentionally excluded — see cell markdown above
query_fields = [
    {"fieldCaption": "Order ID"},
    {"fieldCaption": "Order Date"},
    {"fieldCaption": "Ship Date"},
    {"fieldCaption": "Ship Mode"},
    {"fieldCaption": "Customer Name"},
    {"fieldCaption": "Segment"},
    {"fieldCaption": "Country/Region"},
    {"fieldCaption": "City"},
    {"fieldCaption": "State/Province"},
    {"fieldCaption": "Postal Code"},
    {"fieldCaption": "Region"},
    {"fieldCaption": "Product Name"},
    {"fieldCaption": "Category"},
    {"fieldCaption": "Sub-Category"},
    {"fieldCaption": "Sales"},
    {"fieldCaption": "Quantity"},
    {"fieldCaption": "Discount"},
    {"fieldCaption": "Profit"},
    {"fieldCaption": "Profit Ratio"},
]

query_response = requests.post(
    f"{BASE}/api/v1/vizql-data-service/query-datasource",
    json={
        "datasource": {"datasourceLuid": datasource_luid},
        "query": {"fields": query_fields},
        "options": {"returnFormat": "OBJECTS"}
    },
    headers=HEADERS
)
query_response.raise_for_status()

rows = query_response.json().get("data", [])
df   = pd.DataFrame(rows)

print(f"✓ Query successful")
print(f"  Rows returned: {len(df)}")
print(f"  Columns:       {len(df.columns)}")
print()
display(df.head(10))

## Cell 6 — Sanitize Column Names for Delta

Delta Lake does not allow special characters in column names — specifically
parentheses, spaces, forward slashes, commas, semicolons, and a few others.

Tableau field names often contain these characters, for example:
- `State/Province` → `State_Province`
- `Country/Region` → `Country_Region`
- `Profit Ratio` → `Profit_Ratio`

This cell cleans all column names before writing to Delta so the write doesn't fail.

In [ ]:
print("Sanitizing column names for Delta compatibility...\n")

def clean_col(name):
    """Replace Delta-incompatible characters with underscores."""
    for ch in ["(", ")", " ", ",", ";", "{", "}", "/", "\\", "\n", "\t", "="]:
        name = name.replace(ch, "_")
    return name.strip("_")

original_cols = list(df.columns)
clean_cols    = [clean_col(c) for c in original_cols]
df.columns    = clean_cols

changes = [(o, c) for o, c in zip(original_cols, clean_cols) if o != c]
if changes:
    print("  Columns renamed:")
    for orig, clean in changes:
        print(f"    {orig}  →  {clean}")
else:
    print("  No column names required changes")

print(f"\n✓ Final column names: {list(df.columns)}")

## Cell 7 — Write to Delta Table in Lakehouse

The final step converts the pandas DataFrame to a Spark DataFrame and writes it to the
attached Fabric Lakehouse as a Delta table.

**What `mode('overwrite')` means:** each run replaces the full table with fresh data
from Tableau. For incremental loads, this would change to `mode('append')` with
appropriate filtering logic in Cell 5.

**`overwriteSchema`** is set to `true` to handle cases where the table schema has
changed between runs — for example when fields are added or removed from the query.

Once written, the Delta table is immediately available via:
- **SQL analytics endpoint** — query with T-SQL directly
- **Power BI** — connect via Fabric semantic model
- **Fabric data agent** — natural language queries over the data (Play 2)
- **Downstream notebooks** — `spark.read.table(TABLE_NAME)`

In [ ]:
print(f"Writing {len(df)} rows to Delta table '{TABLE_NAME}'...\n")

spark_df = spark.createDataFrame(df)
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

verification = spark.sql(f"SELECT COUNT(*) as row_count FROM {TABLE_NAME}")
row_count = verification.collect()[0]["row_count"]

print(f"✓ Delta table written successfully")
print(f"  Table name:   {TABLE_NAME}")
print(f"  Rows written: {row_count}")
print(f"  Columns:      {len(df.columns)}")
print()
print(f"  → Queryable via SQL analytics endpoint")
print(f"  → Available in Power BI via Fabric semantic model")
print(f"  → Ready for Fabric data agent (Play 2)")